In [1]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups
from lvt.philadelphia import (
    tax_year_params, parcel_cache_path, split_zero_building_parcels,
    compute_lycd_land_values,
)

# PROTOTYPE: two refinements over model_lycd.ipynb, not a tracked pipeline notebook.
#   Q2 fix: GMA zone-median pricing stratified by residential vs non-residential context
#           (previously pooled all improved parcel types together per zone).
#   Q1 fix: core residential land share pulled from FHFA's tract-level land-price data
#           instead of a flat 20% (see cities/philadelphia/data/fhfa_land_share_by_tract.csv).
CITY_NAME = 'philadelphia'
STATE_FIPS = '42'
COUNTY_FIPS = '101'
LAND_IMPROVEMENT_RATIO = 4.0

# GMA zone assignment: static reference extracted from OPA's 2025 GMA PDF
# (parcel centroid → L1/L2/L3 zone labels; 17 / 84 / 613 zones)
GMA_PATH = Path('data/parcel_gma_assignment.parquet')

# --- Tax year (see lvt/philadelphia.py; do not hardcode a millage here) ---
TAX_YEAR = int(os.environ.get('LVT_TAX_YEAR', 2026))   # override: LVT_TAX_YEAR=2027
TY = tax_year_params(TAX_YEAR)
MILLAGE = TY.combined_mills
PARCEL_PATH = parcel_cache_path(TAX_YEAR)
MODEL_TYPE = f'split_rate_4to1_lycd_refined_ty{TAX_YEAR}'
EXPORT_SUFFIX = f'_lycd_refined_ty{TAX_YEAR}'
print(TY.describe())

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

TY2026: 0.6159% city + 0.7839% school = 1.3998% (13.998 mills) | city target $891,102,000 (projection) | homestead $100,000


C:\Users\druss\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Step 1: Load parcel data

In [2]:
if not PARCEL_PATH.exists():
    raise FileNotFoundError(
        f'{PARCEL_PATH} not found. Build it with:\n'
        f'    python scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR}\n'
        'The cache is keyed by tax year on purpose — opa_properties_public always carries '
        'the latest assessment year, so an unsuffixed cache makes it easy to model one '
        "year's taxable values against another year's expectations with no visible symptom."
    )
gdf = gpd.read_parquet(PARCEL_PATH)
_required = {'parcel_number', 'taxable_land', 'taxable_building', 'market_value',
             'exempt_land', 'exempt_building', 'pin', 'category_code', 'total_area'}
_missing = _required - set(gdf.columns)
if _missing:
    raise ValueError(
        f'{PARCEL_PATH} is missing columns {sorted(_missing)} — rebuild it with '
        f'scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR} --force'
    )
gdf['parcel_number'] = gdf['parcel_number'].astype(str).str.zfill(9)
print(f'Loaded {len(gdf):,} parcels for TY{TAX_YEAR}')
print(f'  taxable base: ${(gdf["taxable_land"].sum() + gdf["taxable_building"].sum())/1e9:.3f}B')

Loaded 583,249 parcels for TY2026
  taxable base: $152.997B


## Step 2: Compute LYCD land values (shared construction + two refinements)

Land value: GMA-hierarchical LYCD, built by `lvt.philadelphia.compute_lycd_land_values` — the
same function `model_lycd.ipynb`, `model_lycd_post_abatement.ipynb` and
`model_lycd_reassessment.ipynb` use for the base construction (lot-area chain, hierarchical
zone medians, KNN fallback, improved-only market-value cap). See the function's docstring and
`docs/LYCD_LAND_MODEL_ROADMAP.md` for what that base method is and is not.

This prototype layers two refinements on top, via two parameters the function accepts for
exactly this purpose:

- **Q2 — zone-median pricing stratified by residential vs non-residential context**
  (`zone_group_col`). The base method pools every improved parcel type into one zone median,
  so a rowhouse in a mostly-residential zone was priced using the SAME comp as any nearby
  commercial or industrial parcel — pulling the rate toward whichever type happens to be more
  common locally. This computes zone medians per (zone-group, GMA level) cell instead.
  Residential context includes vacant/blighted lots: in Philadelphia's rowhouse fabric these
  are overwhelmingly gaps in residential blocks, not comps to commercial or industrial land,
  and the model's own stated design goal is to preserve their development-potential signal,
  which a commercial-zone comp would suppress.
- **Q1 — core-residential land share from FHFA tract data** (`land_pct_improved` passed as a
  per-parcel share instead of a flat scalar). Single family, small/large multi-family and
  other-residential parcels use the FHFA's own tract-level single-family land-share estimate
  (Davis, Larson, Oliner & Shui WP 19-01, 2024 update) instead of OPA's flat 20%, falling back
  to the citywide FHFA median for tracts FHFA doesn't estimate. Mixed use and all
  non-residential categories keep the flat 20% — FHFA's estimate is calibrated on
  single-family sales and would be an extrapolation there. Vacant parcels are unaffected
  (still 100%).

In [3]:
PIN_AREA_PATH = DATA_DIR / 'parcel_areas_by_pin_current.parquet'

if not PIN_AREA_PATH.exists():
    raise FileNotFoundError(
        f'{PIN_AREA_PATH} not found. Build it with:\n'
        '    python scripts/fetch_dor_parcel_areas.py\n'
        'Do NOT fall back to parcel_areas_by_pin.parquet — those areas are Web Mercator '
        '(inflated ~1.704x at this latitude) and mixing them with OPA total_area puts ~5% of '
        'parcels on a different area scale. See the Step 2 notes above.'
    )

pin_areas = pd.read_parquet(PIN_AREA_PATH)
pin_areas['pin'] = pin_areas['pin'].astype(str).str.strip()
print(f'PIN-area lookup: {len(pin_areas):,} PINs (true ground sqft, Mercator-corrected)')
print(f'  median lot: {pin_areas["pin_area_sqft"].median():,.0f} sqft')

PIN-area lookup: 580,097 PINs (true ground sqft, Mercator-corrected)
  median lot: 1,365 sqft


In [4]:
# --- Prototype refinement: neighborhood-varying land share from FHFA data ---
# Instead of a flat 20% land share for every improved parcel, use the FHFA's own
# tract-level single-family land-share estimate (Davis/Larson/Oliner/Shui WP 19-01,
# 2024 update) for core residential categories. Falls back to the citywide FHFA
# median for the ~13% of tracts FHFA doesn't estimate (non-single-family-dominant).
census_tracts = gpd.read_parquet(DATA_DIR / 'census_tracts.gpq')
fhfa_share = pd.read_csv(DATA_DIR / 'fhfa_land_share_by_tract.csv', dtype={'tract_geoid': str})
citywide_fhfa_median = fhfa_share['fhfa_land_share'].median()

_gdf_proj = gdf.to_crs('EPSG:2272')
_centroids = gpd.GeoDataFrame(
    {'parcel_number': gdf['parcel_number']},
    geometry=_gdf_proj.geometry.centroid, crs='EPSG:2272',
).to_crs(census_tracts.crs)
_tract_join = gpd.sjoin(_centroids, census_tracts[['tract_geoid', 'geometry']], how='left', predicate='within')
_tract_join = _tract_join.drop_duplicates('parcel_number')[['parcel_number', 'tract_geoid']]

gdf = gdf.merge(_tract_join, on='parcel_number', how='left')
gdf = gdf.merge(fhfa_share, on='tract_geoid', how='left')
_n_real_match = int(gdf['fhfa_land_share'].notna().sum())
gdf['fhfa_land_share'] = gdf['fhfa_land_share'].fillna(citywide_fhfa_median)

print(f'FHFA tract land-share match: {_n_real_match:,} parcels ({_n_real_match/len(gdf):.1%}) '
      f'matched a real FHFA tract estimate; rest fall back to citywide median {citywide_fhfa_median:.3f}')

FHFA tract land-share match: 508,674 parcels (87.2%) matched a real FHFA tract estimate; rest fall back to citywide median 0.228


In [5]:
LAND_PCT_IMPROVED = 0.20   # OPA's standard land allocation (non-core-residential / no FHFA data)
CORE_RESIDENTIAL_CODES = {'1', '2', '8', '14'}

# --- Prototype refinement (Q2): stratify zone-median pricing by residential vs
# non-residential context, instead of pooling every improved parcel type together.
# A rowhouse (or a vacant lot) in a mostly-residential zone was previously priced using the
# SAME zone median as any nearby commercial/industrial parcel, pulling the comp toward
# whichever type happens to be more common locally. Residential context includes
# vacant/blighted lots: in Philadelphia's rowhouse fabric these are overwhelmingly gaps in
# residential blocks, not comps to commercial or industrial land, and the model's own stated
# design goal is to preserve their development-potential signal, which a commercial-zone
# comp would suppress.
cat_raw = gdf['category_code'].astype(str).str.strip()
VACANT_CODES = {'6', '12', '13'}
RESIDENTIAL_CONTEXT_CODES = {'1', '2', '3', '8', '14'} | VACANT_CODES
gdf['_zone_group'] = np.where(cat_raw.isin(RESIDENTIAL_CONTEXT_CODES), 'Residential', 'NonResidential')

# --- Prototype refinement (Q1): land allocation per parcel type. Core residential
# categories (single family, small/large multi-family, other residential) use the FHFA
# tract-level land share instead of a flat 20%. Mixed use and all non-residential categories
# keep the flat 20% (FHFA's estimate is calibrated on single-family sales and would be an
# extrapolation there). Vacant parcels are unaffected -- compute_lycd_land_values applies
# land_pct_vacant to them regardless of what this Series holds at their rows.
_is_core_resid = cat_raw.isin(CORE_RESIDENTIAL_CODES)
land_pct_improved = pd.Series(
    np.where(_is_core_resid, gdf['fhfa_land_share'], LAND_PCT_IMPROVED),
    index=gdf.index,
)

if not GMA_PATH.exists():
    raise FileNotFoundError(
        f'{GMA_PATH} not found. This is the static GMA zone-assignment reference file; '
        'see model_lycd_reassessment.ipynb Step 2 or CLAUDE.md for how it was built.'
    )
gma = pd.read_parquet(GMA_PATH)

lycd = compute_lycd_land_values(
    gdf, gma, pin_areas,
    land_pct_improved=land_pct_improved,
    zone_group_col='_zone_group',
)
gdf = lycd.gdf
print(lycd.describe())
print(f"  lot area KNN-imputed: {lycd.diagnostics['n_area_knn']:,} parcels")
print(f"  land value KNN-imputed (no GMA zone): {lycd.diagnostics['n_land_knn']:,} parcels "
      "-- a spatial smooth of a neighbour's dollar land value, not this parcel's own zone "
      "rate x area; see docs/LYCD_LAND_MODEL_ROADMAP.md.")

Lot area source: opa_total_area=550,807, knn=30,696, pin_dor=1,351, pin_override=395
  OPA records overridden by surveyed polygon: 395
  total lot area = 0.84x the city (expect <1.0)
GMA assignment: 527,364 matched (90.4%); L3=518,012, knn=55,885, L2=8,791, L1=561
Market-value cap (improved only): 15,486 parcels, $127.94B -> $111.84B (12.6% removed)
  lot area KNN-imputed: 30,696 parcels
  land value KNN-imputed (no GMA zone): 55,885 parcels -- a spatial smooth of a neighbour's dollar land value, not this parcel's own zone rate x area; see docs/LYCD_LAND_MODEL_ROADMAP.md.


## Step 3: Summarize LYCD land base

In [6]:
total_lycd_land = gdf['lycd_land_value'].sum()
total_opa_land  = gdf['taxable_land'].sum()
print(f'Total LYCD land base:    ${total_lycd_land/1e9:.2f}B')
print(f'Total OPA taxable land:  ${total_opa_land/1e9:.2f}B')
print(f'Ratio LYCD/OPA:          {total_lycd_land/total_opa_land:.2f}x')
print()
print('LYCD land value by GMA level (median $):')
print(gdf.groupby('gma_level')['lycd_land_value'].median().sort_values(ascending=False).to_string())
print()
print('LYCD land value percentiles (all parcels):')
for p in [10, 25, 50, 75, 90, 99]:
    v = gdf['lycd_land_value'].quantile(p/100)
    print(f'  p{p:2d}: ${v:,.0f}')


Total LYCD land base:    $111.84B
Total OPA taxable land:  $43.00B
Ratio LYCD/OPA:          2.60x

LYCD land value by GMA level (median $):
gma_level
L1     226200.000000
L2      92041.496673
knn     73617.205391
L3      51159.163526

LYCD land value percentiles (all parcels):
  p10: $19,636
  p25: $30,419
  p50: $52,676
  p75: $94,625
  p90: $185,472
  p99: $1,230,599


## Step 4: Categorize parcels (same overrides as OPA model)

In [7]:
gdf['category_code'] = (
    pd.to_numeric(gdf['category_code'], errors='coerce')
    .astype('Int64')
    .astype(str)
)

CATEGORY_MAP = {
    '1':  'Single Family Residential',
    '2':  'Small Multi-Family (2-4 units)',
    '3':  'Mixed Use',
    '4':  'Commercial',
    '5':  'Industrial',
    '6':  'Vacant Land',
    '7':  'Other Commercial',
    '8':  'Other Residential',
    '9':  'Hotel',
    '10': 'Office / Commercial Condo',
    '11': 'Other',
    '12': 'Vacant Land',
    '13': 'Vacant Land',
    '14': 'Large Multi-Family (5+ units)',
    '15': 'Retail / General Commercial',
}
gdf['PROPERTY_CATEGORY'] = gdf['category_code'].map(CATEGORY_MAP).fillna('Other')

# Override 1: $0 improvement -> Vacant Land
gdf.loc[gdf['taxable_building'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Override 2: a $0 taxable building line has three different causes, and calling all of
# them "abated" put ~13K homesteaded rowhomes in the abated bucket -- then revoked their
# Homestead Exemption under the reform. Split on the year's statutory homestead cap.
GENUINE_VACANT_CODES = {'6', '12', '13'}
_zb = split_zero_building_parcels(
    gdf, gdf['PROPERTY_CATEGORY'], TY.homestead_exemption, CATEGORY_MAP,
    genuine_vacant_codes=tuple(GENUINE_VACANT_CODES),
)
gdf['PROPERTY_CATEGORY'] = _zb.category
abated_mask = _zb.abated
print(_zb.describe())

# Override 3: OPA-vacant with nonzero building value
improved_vacant_mask = (
    gdf['category_code'].isin(GENUINE_VACANT_CODES) &
    (gdf['taxable_building'] > 0)
)
gdf.loc[improved_vacant_mask, 'PROPERTY_CATEGORY'] = 'Improved Vacant Land'

gdf['taxable_total'] = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0)
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

# Override 4: fully exempt parcels
EXEMPT_CATEGORY_MAP = {k: v + ' â€” Exempt' for k, v in CATEGORY_MAP.items()}
exempt_mask = gdf['full_exmp'] == 1
gdf.loc[exempt_mask, 'PROPERTY_CATEGORY'] = (
    gdf.loc[exempt_mask, 'category_code']
    .map(EXEMPT_CATEGORY_MAP)
    .fillna('Other â€” Exempt')
)

print(f'Total parcels: {len(gdf):,}')
print(f'Fully exempt: {gdf["full_exmp"].sum():,}  |  '
      f'Abated: {abated_mask.sum():,}  |  '
      f'Improved vacant: {improved_vacant_mask.sum():,}  |  '
      f'Taxable: {(gdf["full_exmp"] == 0).sum():,}')
print()
print('Property category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts().to_string())

zero-building line: 14,287 abated | 13,995 homestead-zeroed (96.3% confirmed by OPA's homestead flag) | 1,119 genuinely $0 improvement


Total parcels: 583,249


Fully exempt: 36,932  |  Abated: 14,287  |  Improved vacant: 880  |  Taxable: 546,317

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential                    430570
Small Multi-Family (2-4 units)                38685
Vacant Land                                   30557
Single Family Residential â€” Exempt          20078
Abated / Construction Exemption               14287
Mixed Use                                     13743
Vacant Land â€” Exempt                        11714
Commercial                                     8802
Industrial                                     3553
Commercial â€” Exempt                          3298
Large Multi-Family (5+ units)                  3002
Other Residential                              1108
Small Multi-Family (2-4 units) â€” Exempt      1026
Improved Vacant Land                            880
Office / Commercial Condo                       825
Large Multi-Family (5+ units) â€” Exempt        264
Industrial â€” Exempt          

## Step 5: Current tax (OPA taxable values — revenue baseline)

In [8]:
gdf['millage_rate'] = MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

city_revenue = gdf['taxable_total'].mul(TY.city_mills / 1000).sum()

print(f'Modeled combined levy (city + school):  ${current_revenue:,.0f}')
print(f'Implied city-only portion ({TY.city_rate_pct}%):   ${city_revenue:,.0f}')

if TY.city_revenue_target is None:
    # TY2027: bills are not due until March 2027, so there are no collections to check
    # against. This run is a forward-looking scenario, not a validated baseline.
    print(f'\nNO REVENUE VALIDATION for TY{TAX_YEAR}.')
    print(f'  {TY.source}')
else:
    gap_pct = (city_revenue / TY.city_revenue_target - 1) * 100
    print(f'City-only target ({TY.target_kind}):            ${TY.city_revenue_target:,}')
    print(f'City portion gap: {gap_pct:+.2f}%  (expected: a few % over, from delinquency)')
    assert abs(gap_pct) < 10.0, (
        f'City gap {gap_pct:.2f}% exceeds 10% for TY{TAX_YEAR}. Check that the assessment '
        f'year, the City rate ({TY.city_rate_pct}%) and the revenue target all refer to the '
        'same billing year — see lvt/philadelphia.py.'
    )

Modeled combined levy (city + school):  $2,141,653,043
Implied city-only portion (0.6159%):   $942,308,979
City-only target (projection):            $891,102,000
City portion gap: +5.75%  (expected: a few % over, from delinquency)


## Step 6: Build GMA LYCD reform base

Land value: GMA hierarchical LYCD (`lycd_land_value`).
Building value: OPA `taxable_building` for non-abated parcels (post-exemption, preserves
Homestead and other reliefs).

For abated parcels (OPA shows zero taxable_building due to active 10-year construction
abatements): use OPA's `exempt_building` — the assessed building value that the abatement
shields from taxation — as `model_building`. For the ~2,100 parcels where `exempt_building`
is also zero (building not yet assessed, mid-construction), fall back to
`market_value − taxable_land` as the implied building value.

In [9]:
gdf['model_land']     = gdf['lycd_land_value'].clip(lower=0)
gdf['model_building'] = pd.to_numeric(gdf['taxable_building'], errors='coerce').fillna(0).clip(lower=0)

abated = gdf['PROPERTY_CATEGORY'] == 'Abated / Construction Exemption'

# OPA's actual assessed building value (shielded from tax by the abatement)
exempt_bldg  = pd.to_numeric(gdf['exempt_building'], errors='coerce').fillna(0)
market_val   = pd.to_numeric(gdf['market_value'],    errors='coerce').fillna(0)
tax_land     = pd.to_numeric(gdf['taxable_land'],     errors='coerce').fillna(0)

# Fallback for ~2,100 mid-construction parcels where exempt_building = 0
implied_bldg = (market_val - tax_land).clip(lower=0)
abated_bldg  = exempt_bldg.where(exempt_bldg > 0, implied_bldg)

gdf.loc[abated, 'model_building'] = abated_bldg[abated].values

n_exempt_bldg = int((abated & (exempt_bldg > 0)).sum())
n_fallback    = int((abated & (exempt_bldg <= 0)).sum())
print(f'Abated parcels using exempt_building:        {n_exempt_bldg:,}')
print(f'Abated parcels using market_value fallback:  {n_fallback:,}')
print()
print(f'Reform land base:          ${gdf["model_land"].sum()/1e9:.2f}B')
print(f'Reform improvement base:   ${gdf["model_building"].sum()/1e9:.2f}B')
print(f'  of which abated bldg:    ${gdf.loc[abated,"model_building"].sum()/1e9:.2f}B')
print(f'OPA taxable land base:     ${pd.to_numeric(gdf["taxable_land"],errors="coerce").sum()/1e9:.2f}B')
print(f'OPA taxable building base: ${pd.to_numeric(gdf["taxable_building"],errors="coerce").sum()/1e9:.2f}B')


Abated parcels using exempt_building:        14,269
Abated parcels using market_value fallback:  18

Reform land base:          $111.84B
Reform improvement base:   $126.21B
  of which abated bldg:    $16.21B
OPA taxable land base:     $43.00B
OPA taxable building base: $110.00B


## Step 7: Revenue-neutral split-rate model (4:1 land:improvement)

In [10]:
taxable = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='model_land',
    improvement_value_col='model_building',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine exempt parcels
exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
exempt['taxable_land_value'] = 0.0
exempt['taxable_improvement_value'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f'Land millage:        {land_millage:.4f} mills')
print(f'Improvement millage: {improvement_millage:.4f} mills')
print(f'Revenue check:       ${new_revenue:,.0f} (target: ${taxable["current_tax"].sum():,.0f})')
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Philadelphia â€” 4:1 Split-Rate Tax Impact (LYCD Land Values)')

Land millage:        21.3310 mills
Improvement millage: 5.3328 mills
Revenue check:       $2,141,653,043 (target: $2,141,653,043)




Philadelphia â€” 4:1 Split-Rate Tax Impact (LYCD Land Values)
                                 Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
                Single Family Residential 430570   $-230,169,849      -18.6%      $-535        $-507    9.2%     -24.2%            18.3%            68.7%
           Small Multi-Family (2-4 units)  38685    $-75,089,491      -29.7%    $-1,941      $-1,346  -27.2%     -35.1%             7.4%            84.8%
                              Vacant Land  30557    $407,277,540      779.0%    $13,328       $2,378 1264.8%     544.1%            96.7%             2.5%
     Single Family Residential â€” Exempt  20078              $0        0.0%         $0           $0    0.0%       0.0%             0.0%             0.0%
          Abated / Construction Exemption  14287    $101,540,827      276.0%     $7,107       $2,517  306.4%     179.9%            99.7%             0.2%
             

## Step 8: Census join

In [11]:
import concurrent.futures

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f'Census join: {gdf["std_geoid"].notna().mean()*100:.1f}% matched')
        except concurrent.futures.TimeoutError:
            print('Census API timed out â€” skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


In [12]:
out_df = save_standard_export(
    df=gdf,
    city=f'{CITY_NAME}{EXPORT_SUFFIX}',
    output_path=f'../../analysis/data/{CITY_NAME}{EXPORT_SUFFIX}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    parcel_id_col='parcel_number',
)
print('Done.')

  [warn] philadelphia_lycd_refined_ty2026: non-standard property categories (will be preserved): ['Abated / Construction Exemption', 'Commercial â€” Exempt', 'Hotel â€” Exempt', 'Improved Vacant Land', 'Industrial â€” Exempt', 'Large Multi-Family (5+ units) â€” Exempt', 'Mixed Use â€” Exempt', 'Office / Commercial Condo â€” Exempt', 'Other Commercial â€” Exempt', 'Other Residential â€” Exempt', 'Other â€” Exempt', 'Retail / General Commercial â€” Exempt', 'Single Family Residential â€” Exempt', 'Small Multi-Family (2-4 units) â€” Exempt', 'Vacant Land â€” Exempt']


  ✓ philadelphia_lycd_refined_ty2026: 583,249 rows → ../../analysis/data/philadelphia_lycd_refined_ty2026.csv  [model: split_rate_4to1_lycd_refined_ty2026]
Done.
